In [1]:
#%% CELL: setup
from pathlib import Path
import json, sys, types, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             balanced_accuracy_score, roc_auc_score, average_precision_score,
                             roc_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 42
OUT_DIR = Path("outputs_ablation")
OUT_DIR.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")
DATA_FILE = Path("brfss_2023_selected.csv")
assert DATA_FILE.exists(), "run the replication notebook first to download brfss_2023_selected.csv"
print("scipy", stats.__name__, "| output dir", OUT_DIR.resolve())


scipy scipy.stats | output dir C:\Users\akbar\penelitian\my-research\notebook-replikasi\outputs_ablation


In [2]:
#%% CELL: load + recode
VARIABLES = [
    "DIABETE4","SEXVAR","_RACE","_AGEG5YR","EDUCA","_INCOMG1","INCOME3","EMPLOY1","MARITAL",
    "_BMI5CAT","GENHLTH","PHYSHLTH","MENTHLTH","BPHIGH6","TOLDHI3","HAVARTH4",
    "CVDINFR4","CVDSTRK3","CVDCRHD4","EXERANY2","SMOKDAY2","ALCDAY4","_HLTHPL1",
    "BLIND","DEAF","DECIDE","DIFFDRES","DIFFWALK",
]
raw = pd.read_csv(DATA_FILE, low_memory=False)
print(f"Rows national BRFSS 2023: {len(raw):,} | cols {raw.shape[1]}")

work = raw.loc[raw["DIABETE4"].isin([1, 2, 3, 4])].copy()
work["diabetes"] = work["DIABETE4"].isin([1, 2]).astype(int)

predictors = [c for c in VARIABLES if c != "DIABETE4"]
RESPONSE_CODES = [7, 9, 77, 88, 99, 777, 888, 999, 7777, 9999]
for col in predictors:
    work[col] = pd.to_numeric(work[col], errors="coerce")
    work.loc[work[col].isin(RESPONSE_CODES), col] = np.nan

TX_COLS = ["BPHIGH6", "DIFFWALK", "DEAF", "BLIND", "DECIDE", "DIFFDRES",
           "CVDINFR4", "CVDSTRK3", "CVDCRHD4", "HAVARTH4", "MEDCOST1"]
comorbidity_cols = ["BPHIGH6","TOLDHI3","HAVARTH4","CVDINFR4","CVDSTRK3","CVDCRHD4"]
disability_cols = ["BLIND","DEAF","DECIDE","DIFFDRES","DIFFWALK"]
work["comorbidity_count"] = work[comorbidity_cols].eq(1).sum(axis=1)
work["disability_count"] = work[disability_cols].eq(1).sum(axis=1)

PAPER_SCALE_N = 5_634
if len(work) > PAPER_SCALE_N:
    _, work = train_test_split(work, test_size=PAPER_SCALE_N/len(work),
                              stratify=work["diabetes"], random_state=RANDOM_STATE)
    work = work.copy()

print(f"Analytical cohort: {len(work):,}")
print(pd.DataFrame({"n": work["diabetes"].value_counts(),
                    "share": work["diabetes"].value_counts(normalize=True).round(4)}).sort_index())
print("\nNaN share of TX proxies inside cohort:")
print(work[TX_COLS].isna().mean().round(4).sort_values(ascending=False).to_string())


Rows national BRFSS 2023: 433,323 | cols 30


Analytical cohort: 5,634
             n   share
diabetes              
0         4813  0.8543
1          821  0.1457

NaN share of TX proxies inside cohort:
DIFFWALK    0.0403
DECIDE      0.0399
DIFFDRES    0.0399
BLIND       0.0346
DEAF        0.0332
CVDCRHD4    0.0078
CVDINFR4    0.0053
CVDSTRK3    0.0043
HAVARTH4    0.0043
BPHIGH6     0.0035
MEDCOST1    0.0000


In [3]:
#%% CELL: build three feature sets (A = paper, B = no leakage, C = B + ordinal recode)

# --- Model C ordinal recodes, derived from RAW so the "88 = none" answer is not lost.
# ALCDAY4: 1xx = days/week, 2xx = days/month, 888 = none, 777 = refused.
alc = raw.loc[work.index, "ALCDAY4"]
days_month = pd.Series(np.nan, index=alc.index)
days_month[alc.between(101, 107)] = (alc[alc.between(101, 107)] - 100) * 4.345
days_month[alc.between(201, 231)] = alc[alc.between(201, 231)] - 200
days_month[alc == 888] = 0.0
work["alc_days_month"] = days_month

# MENTHLTH / PHYSHLTH: 1-30 days, 88 = none, 77/99 = refused/don't know.
for src, dst in [("MENTHLTH", "mental_days"), ("PHYSHLTH", "physical_days")]:
    v = raw.loc[work.index, src]
    v = v.where(v.between(1, 30), np.where(v == 88, 0.0, np.nan))
    work[dst] = v

print("ordinal recode sanity (mean / %zero / %nan):")
for c in ["alc_days_month", "mental_days", "physical_days"]:
    s = work[c]
    print(f"  {c:16s} {s.mean():7.3f}  {(s==0).mean():.3f}  {s.isna().mean():.3f}")

BASE = [p for p in predictors if p not in ("ALCDAY4", "MENTHLTH", "PHYSHLTH")]
SETS = {
    "A_paper":      dict(cat=BASE + ["ALCDAY4", "MENTHLTH", "PHYSHLTH"], num=[]),
    # B: A minus post-diagnosis proxies ONLY (clean leakage ablation vs A)
    "B_no_leakage": dict(cat=[c for c in BASE if c not in TX_COLS] + ["ALCDAY4", "MENTHLTH", "PHYSHLTH"], num=[]),
    # C: B with the 3 count variables recoded ordinal (clean encoding ablation vs B)
    "C_ordinal":    dict(cat=[c for c in BASE if c not in TX_COLS],
                         num=["alc_days_month", "mental_days", "physical_days"]),
}
for k, v in SETS.items():
    print(f"{k:14s} categorical={len(v['cat']):2d} numeric={len(v['num'])}")
print("\nTX proxies removed in B and C:", TX_COLS)


ordinal recode sanity (mean / %zero / %nan):
  alc_days_month     5.085  0.439  0.069
  mental_days        4.241  0.614  0.019
  physical_days      4.621  0.591  0.029
A_paper        categorical=27 numeric=0
B_no_leakage   categorical=17 numeric=0
C_ordinal      categorical=14 numeric=3

TX proxies removed in B and C: ['BPHIGH6', 'DIFFWALK', 'DEAF', 'BLIND', 'DECIDE', 'DIFFDRES', 'CVDINFR4', 'CVDSTRK3', 'CVDCRHD4', 'HAVARTH4', 'MEDCOST1']


In [4]:
#%% CELL: split, preprocessor builder, model factory

def make_preprocessor(cat_cols, num_cols):
    parts = []
    if cat_cols:
        parts.append(("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), cat_cols))
    if num_cols:
        parts.append(("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
        ]), num_cols))
    return ColumnTransformer(parts, remainder="drop", verbose_feature_names_out=False)


def make_models():
    return {
        "Logistic Regression": LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),
        "SVM": SVC(probability=True, random_state=RANDOM_STATE),
        "KNN": KNeighborsClassifier(),
        "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE),
        "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                 subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
                                 n_jobs=-1, random_state=RANDOM_STATE),
    }

y = work["diabetes"].copy()
# One split for all three feature sets, so models are compared on identical rows.
train_idx, test_idx = train_test_split(work.index, test_size=0.20, stratify=y,
                                       random_state=RANDOM_STATE)
assert set(train_idx).isdisjoint(set(test_idx))
print(f"Train {len(train_idx):,} | Test {len(test_idx):,} | test prevalence {y.loc[test_idx].mean():.4f}")


Train 4,507 | Test 1,127 | test prevalence 0.1455


In [5]:
#%% CELL: fit 3 feature sets x 7 models (SMOTE inside the training fold only)

import pickle

rows, proba_store, fitted = [], {}, {}
y_test = y.loc[test_idx]

for set_name, spec in SETS.items():
    pre = make_preprocessor(spec["cat"], spec["num"])
    Xtr = pre.fit_transform(work.loc[train_idx, spec["cat"] + spec["num"]])
    Xte = pre.transform(work.loc[test_idx, spec["cat"] + spec["num"]])
    feat = pre.get_feature_names_out()

    smote = SMOTE(random_state=RANDOM_STATE)
    Xtr_s, ytr_s = smote.fit_resample(Xtr, y.loc[train_idx])
    assert len(Xte) == len(y_test)

    for name, model in make_models().items():
        model.fit(Xtr_s, ytr_s)
        p = model.predict_proba(Xte)[:, 1]
        pred = (p >= 0.5).astype(int)
        rows.append({
            "FeatureSet": set_name, "Model": name, "n_features": Xtr.shape[1],
            "Accuracy": accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred, zero_division=0),
            "Recall": recall_score(y_test, pred, zero_division=0),
            "F1": f1_score(y_test, pred, zero_division=0),
            "Balanced Accuracy": balanced_accuracy_score(y_test, pred),
            "AUROC": roc_auc_score(y_test, p),
            "PR-AUC": average_precision_score(y_test, p),
        })
        proba_store[(set_name, name)] = p
        fitted[(set_name, name)] = (model, feat)
    print(f"done {set_name}: {len(Xtr)}x{Xtr.shape[1]} -> {len(Xtr_s)} after SMOTE")

results = pd.DataFrame(rows)
results.to_csv(OUT_DIR / "metrics_by_featureset.csv", index=False)
with open(OUT_DIR / "state.pkl", "wb") as f:
    pickle.dump({"proba": proba_store, "y_test": y_test, "results": results,
                 "fitted_feat": {k: v[1] for k, v in fitted.items()}}, f)
print(results.groupby("FeatureSet")[["AUROC", "PR-AUC", "Recall"]].max().round(4).to_string())


  File "C:\Users\akbar\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\akbar\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 546, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\akbar\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1022, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\akbar\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1491, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


done A_paper: 4507x183 -> 7700 after SMOTE


done B_no_leakage: 4507x161 -> 7700 after SMOTE


done C_ordinal: 4507x76 -> 7700 after SMOTE
               AUROC  PR-AUC  Recall
FeatureSet                          
A_paper       0.7816  0.4043  0.7561
B_no_leakage  0.7817  0.4046  0.7073
C_ordinal     0.7877  0.3989  0.7256


In [6]:
#%% CELL: DeLong (positives-first, self-checked vs sklearn) + McNemar + Bonferroni

def _midrank(x):
    order = np.argsort(x, kind="mergesort"); z = x[order]; n = len(x)
    t = np.zeros(n); i = 0
    while i < n:
        j = i
        while j < n and z[j] == z[i]:
            j += 1
        t[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(n); out[order] = t
    return out


def delong_test(y_true, p1, p2):
    """DeLong (1988) for two correlated AUROCs. The fast algorithm needs positives first."""
    y_true = np.asarray(y_true).astype(int)
    pos, neg = np.flatnonzero(y_true == 1), np.flatnonzero(y_true == 0)
    m, n = len(pos), len(neg)
    preds = np.vstack([p1, p2])[:, np.concatenate([pos, neg])]
    tx = np.array([_midrank(r[:m]) for r in preds])
    ty = np.array([_midrank(r[m:]) for r in preds])
    tz = np.array([_midrank(r) for r in preds])
    auc = tz[:, :m].sum(1) / (m * n) - (m + 1) / (2 * n)
    cov = np.cov((tz[:, :m] - tx) / n) / m + np.cov(1 - (tz[:, m:] - ty) / m) / n
    z = (auc[0] - auc[1]) / np.sqrt(cov[0, 0] + cov[1, 1] - 2 * cov[0, 1])
    assert abs(auc[0] - roc_auc_score(y_true, p1)) < 1e-9, "DeLong AUC != sklearn AUC"
    return auc[0], auc[1], z, 2 * stats.norm.sf(abs(z))


def mcnemar_test(y_true, pred1, pred2):
    b = int(np.sum((pred1 == y_true) & (pred2 != y_true)))
    c = int(np.sum((pred1 != y_true) & (pred2 == y_true)))
    if b + c == 0:
        return b, c, np.nan
    return b, c, stats.chi2.sf((abs(b - c) - 1) ** 2 / (b + c), df=1)


comparisons = []
for a_set, b_set in [("A_paper", "B_no_leakage"), ("B_no_leakage", "C_ordinal"), ("A_paper", "C_ordinal")]:
    for model in make_models():
        pA, pB = proba_store[(a_set, model)], proba_store[(b_set, model)]
        aA, aB, z, p = delong_test(y_test.values, pA, pB)
        b, c, pm = mcnemar_test(y_test.values, (pA >= .5).astype(int), (pB >= .5).astype(int))
        comparisons.append({"compare": f"{a_set} vs {b_set}", "Model": model, "AUROC_1": aA,
                            "AUROC_2": aB, "delta": aA - aB, "DeLong_z": z, "DeLong_p": p,
                            "McNemar_b": b, "McNemar_c": c, "McNemar_p": pm})
cmp_df = pd.DataFrame(comparisons)
cmp_df["DeLong_p_bonf"] = np.minimum(cmp_df["DeLong_p"] * len(cmp_df), 1.0)
cmp_df.to_csv(OUT_DIR / "delong_mcnemar.csv", index=False)
pd.set_option("display.width", 180)
print(f"{len(cmp_df)} tests, Bonferroni-corrected")
print(cmp_df[["compare", "Model", "AUROC_1", "AUROC_2", "delta", "DeLong_p", "DeLong_p_bonf", "McNemar_p"]]
      .round(4).to_string(index=False))


21 tests, Bonferroni-corrected
                  compare               Model  AUROC_1  AUROC_2   delta  DeLong_p  DeLong_p_bonf  McNemar_p
  A_paper vs B_no_leakage Logistic Regression   0.7726   0.7671  0.0055    0.4016         1.0000     0.4305
  A_paper vs B_no_leakage                 SVM   0.7688   0.7649  0.0040    0.6796         1.0000     0.7273
  A_paper vs B_no_leakage                 KNN   0.6639   0.6139  0.0500    0.0252         0.5289     0.0024
  A_paper vs B_no_leakage       Decision Tree   0.5642   0.5987 -0.0345    0.1243         1.0000     0.9485
  A_paper vs B_no_leakage       Random Forest   0.7816   0.7610  0.0206    0.1014         1.0000     0.5677
  A_paper vs B_no_leakage   Gradient Boosting   0.7808   0.7817 -0.0009    0.9191         1.0000     0.1344
  A_paper vs B_no_leakage             XGBoost   0.7691   0.7669  0.0022    0.7942         1.0000     0.5754
B_no_leakage vs C_ordinal Logistic Regression   0.7671   0.7865 -0.0194    0.0146         0.3059     0.68

In [7]:
#%% CELL: 5-fold stratified CV on the full cohort (per feature set, GB only) 

cv_rows = []
for set_name, spec in SETS.items():
    X_all = work[spec["cat"] + spec["num"]]
    y_all = y
    pipe = ImbPipeline([
        ("pre", make_preprocessor(spec["cat"], spec["num"])),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE)),
    ])
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    aucs = cross_val_score(pipe, X_all, y_all, cv=skf, scoring="roc_auc", n_jobs=-1)
    auprcs = cross_val_score(pipe, X_all, y_all, cv=skf, scoring="average_precision", n_jobs=-1)
    cv_rows.append({"FeatureSet": set_name, "AUROC_mean": aucs.mean(), "AUROC_sd": aucs.std(ddof=1),
                    "PR-AUC_mean": auprcs.mean(), "PR-AUC_sd": auprcs.std(ddof=1),
                    "folds_AUROC": np.round(aucs, 4).tolist()})
    print(f"{set_name:14s} AUROC {aucs.mean():.4f} +/- {aucs.std(ddof=1):.4f} | "
          f"PR-AUC {auprcs.mean():.4f} +/- {auprcs.std(ddof=1):.4f}")

cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv(OUT_DIR / "cv5_gradient_boosting.csv", index=False)
print(cv_df[["FeatureSet", "AUROC_mean", "AUROC_sd", "PR-AUC_mean"]].round(4).to_string(index=False))


A_paper        AUROC 0.7871 +/- 0.0243 | PR-AUC 0.3882 +/- 0.0333


B_no_leakage   AUROC 0.7714 +/- 0.0245 | PR-AUC 0.3610 +/- 0.0365


C_ordinal      AUROC 0.7852 +/- 0.0240 | PR-AUC 0.3867 +/- 0.0368
  FeatureSet  AUROC_mean  AUROC_sd  PR-AUC_mean
     A_paper      0.7871    0.0243       0.3882
B_no_leakage      0.7714    0.0245       0.3610
   C_ordinal      0.7852    0.0240       0.3867


In [8]:
#%% CELL: SHAP for the best feature set — top-feature drift A -> B -> C

if "torch" not in sys.modules:
    sys.modules["torch"] = types.ModuleType("torch")
import shap

rng = np.random.default_rng(RANDOM_STATE)
sub = rng.choice(len(y_test), size=min(1000, len(y_test)), replace=False)
y_sub = y_test.values[sub]

shap_top = {}
for set_name, spec in SETS.items():
    model, feat = fitted[(set_name, "Gradient Boosting")]
    pre = make_preprocessor(spec["cat"], spec["num"]).fit(work.loc[train_idx, spec["cat"] + spec["num"]])
    Xte = pre.transform(work.loc[test_idx, spec["cat"] + spec["num"]])[sub]
    vals = shap.TreeExplainer(model).shap_values(Xte)
    if isinstance(vals, list):
        vals = vals[1]
    mean_abs = np.abs(vals).mean(axis=0)
    order = np.argsort(mean_abs)[::-1][:15]
    shap_top[set_name] = [(feat[i], float(mean_abs[i])) for i in order]
    plt.figure(figsize=(9, 6))
    shap.summary_plot(vals, Xte, feature_names=feat, max_display=15, show=False)
    plt.title(f"SHAP — {set_name} (Gradient Boosting)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"shap_{set_name}.png", dpi=170, bbox_inches="tight")
    plt.close()
    print(f"\n{set_name} top-5:")
    for f_, v_ in shap_top[set_name][:5]:
        print(f"   {f_:28s} {v_:.4f}")

json.dump({k: v for k, v in shap_top.items()}, open(OUT_DIR / "shap_top_features.json", "w"), indent=1)
print("\nsaved shap_top_features.json")



A_paper top-5:
   BPHIGH6_1.0                  0.5133
   ALCDAY4_201.0                0.3046
   GENHLTH_1.0                  0.2131
   _BMI5CAT_4.0                 0.1837
   TOLDHI3_1.0                  0.1729



B_no_leakage top-5:
   ALCDAY4_201.0                0.3271
   GENHLTH_1.0                  0.2760
   _BMI5CAT_4.0                 0.2530
   GENHLTH_2.0                  0.2232
   _AGEG5YR_11.0                0.2079



C_ordinal top-5:
   TOLDHI3_1.0                  0.2810
   GENHLTH_1.0                  0.2621
   _BMI5CAT_4.0                 0.2373
   _AGEG5YR_11.0                0.2331
   alc_days_month               0.2265

saved shap_top_features.json


In [9]:
#%% CELL: figures + reproducibility record

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_df = results.melt(id_vars=["FeatureSet", "Model"], value_vars=["AUROC", "PR-AUC"],
                       var_name="Metric", value_name="Score")
sns.barplot(data=plot_df, x="Score", y="Model", hue="FeatureSet", ax=axes[0])
axes[0].set_title("Held-out AUROC / PR-AUC by feature set")
axes[0].set_xlim(0.5, 0.9)
axes[0].legend(fontsize=8, title=None)

for set_name in SETS:
    p = proba_store[(set_name, "Gradient Boosting")]
    fpr, tpr, _ = roc_curve(y_test, p)
    axes[1].plot(fpr, tpr, label=f"GB {set_name} (AUC {roc_auc_score(y_test, p):.4f})")
axes[1].plot([0, 1], [0, 1], "k--", alpha=.5)
axes[1].set_xlabel("False positive rate"); axes[1].set_ylabel("True positive rate")
axes[1].set_title("ROC — Gradient Boosting, three feature sets")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "ablation_overview.png", dpi=170, bbox_inches="tight")
plt.close()
print("saved ablation_overview.png")

meta = {
    "paper_doi": "10.1177/21501319251400546",
    "reference_for_design": "10.1186/s12889-026-27570-3 (Noh et al. 2026, Model A/B post-diagnosis leakage)",
    "dataset": "CDC BRFSS 2023 national public-use extract, deterministic stratified paper-scale proxy n=5,634",
    "feature_sets": {k: {"categorical": v["cat"], "numeric": v["num"]} for k, v in SETS.items()},
    "dropped_post_diagnosis_proxies": TX_COLS,
    "split": "single 80:20 stratified split, random_state=42, identical rows across feature sets",
    "resampling": "SMOTE inside the training fold only; test set never resampled",
    "statistics": ["DeLong (1988) AUROC test", "McNemar continuity-corrected", "5-fold stratified CV (GB)"],
    "caveat": "Proxy cohort, not Tennessee. Ordinal recodes for ALCDAY4/MENTHLTH/PHYSHLTH are an interpretation of BRFSS codebook semantics, not verified against the paper.",
}
(OUT_DIR / "reproducibility_ablation.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print(json.dumps(meta, indent=2)[:900])


saved ablation_overview.png
{
  "paper_doi": "10.1177/21501319251400546",
  "reference_for_design": "10.1186/s12889-026-27570-3 (Noh et al. 2026, Model A/B post-diagnosis leakage)",
  "dataset": "CDC BRFSS 2023 national public-use extract, deterministic stratified paper-scale proxy n=5,634",
  "feature_sets": {
    "A_paper": {
      "categorical": [
        "SEXVAR",
        "_RACE",
        "_AGEG5YR",
        "EDUCA",
        "_INCOMG1",
        "INCOME3",
        "EMPLOY1",
        "MARITAL",
        "_BMI5CAT",
        "GENHLTH",
        "BPHIGH6",
        "TOLDHI3",
        "HAVARTH4",
        "CVDINFR4",
        "CVDSTRK3",
        "CVDCRHD4",
        "EXERANY2",
        "SMOKDAY2",
        "_HLTHPL1",
        "BLIND",
        "DEAF",
        "DECIDE",
        "DIFFDRES",
        "DIFFWALK",
        "ALCDAY4",
        "MENTHLTH",
        "PHYSHLTH"
      ],
      "numeric": []
    },
    "B_no_leakage": {
  
